# RAG Pipeline

**Goal:** Build a complete Retrieval-Augmented Generation (RAG) pipeline to provide candidates with intelligent, grounded feedback based on ATS guidelines and resume best practices.

## Why this module exists
Instead of generic AI advice, a real hiring assistant must ground its answers in domain-specific HR knowledge. The RAG pipeline ensures that the candidate's feedback is accurate, actionable, and aligned with company standards.

## How it integrates
It consumes a custom Knowledge Base (ATS guidelines, etc.) and uses FAISS to retrieve relevant chunks when a candidate asks a question (in# ). The context is then passed to an LLM (Gemini) to generate the final response.

## Algorithms used
- **Document Chunking**: LangChain's RecursiveCharacterTextSplitter.
- **Embeddings**: `sentence-transformers` (`all-MiniLM-L6-v2`) for fast, local embedding generation.
- **Vector Store**: FAISS (Facebook AI Similarity Search) for efficient similarity search.
- **Generative AI**: `google-generativeai` (Gemini) for grounded text generation.

## Inputs & Outputs
- **Input**: A corpus of HR guidelines, candidate queries.
- **Output**: FAISS index, Prompt Templates, Grounded Responses.


In [ ]:
import os
import json
import numpy as np
import faiss
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import google.generativeai as genai

# Ensure directories exist
os.makedirs('../knowledge_base', exist_ok=True)
os.makedirs('../vector_store/faiss_index', exist_ok=True)
os.makedirs('../reports/Module_7', exist_ok=True)

# Try to configure Gemini API
api_key = os.getenv("GEMINI_API_KEY")
if api_key:
    genai.configure(api_key=api_key)
    print("Gemini API configured.")
else:
    print("Warning: GEMINI_API_KEY environment variable not found. The generation step will fail or be bypassed.")


## 1. Create the Knowledge Base

In [ ]:
knowledge_base = [
    {
        "title": "ATS Formatting Guidelines",
        "content": "To pass the ATS (Applicant Tracking System), candidates must avoid using complex tables, columns, or images. Use standard fonts like Arial or Calibri. Keep the layout simple and save the resume as a PDF unless specified otherwise. Use standard headings like 'Experience', 'Education', and 'Skills'."
    },
    {
        "title": "Action Verbs for Experience",
        "content": "When describing work experience, always start bullet points with strong action verbs such as Developed, Managed, Orchestrated, Designed, or Optimized. Avoid passive phrases like 'Responsible for' or 'Helped with'. Quantify achievements with metrics (e.g., 'Increased revenue by 20%')."
    },
    {
        "title": "Skills Section Best Practices",
        "content": "Tailor the skills section to match the job description. Do not list generic skills like 'Hardworking' or 'Team Player'. Focus on hard skills, tools, and technologies (e.g., Python, AWS, React). Group them logically into categories if there are many."
    },
    {
        "title": "Handling Employment Gaps",
        "content": "Address employment gaps honestly. If you took time off for personal reasons, state it briefly. If you used the time to upskill, mention the courses or certifications completed during that period. Transparency is preferred over unexplained gaps."
    },
    {
        "title": "Contact Information Checklist",
        "content": "Ensure contact information is clearly visible at the top. Include your full name, a professional email address, a phone number, and a link to your LinkedIn profile or GitHub portfolio. Do not include personal details like age, marital status, or photo unless applying in specific regions."
    }
]

kb_path = '../knowledge_base/hr_guidelines.json'
with open(kb_path, 'w') as f:
    json.dump(knowledge_base, f, indent=2)
print(f"Knowledge Base saved to {kb_path}")


## 2. Chunking Documents

In [ ]:
# Extract text from the KB
documents = [doc['title'] + "\n" + doc['content'] for doc in knowledge_base]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

chunks = []
for doc in documents:
    chunks.extend(text_splitter.split_text(doc))
    
print(f"Created {len(chunks)} chunks from {len(documents)} documents.")
for i, chunk in enumerate(chunks[:2]):
    print(f"Chunk {i+1}: {chunk}\n")


## 3. Generate Embeddings & Build FAISS Index

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for chunks
print("Generating embeddings...")
chunk_embeddings = embedding_model.encode(chunks)

# Build FAISS index
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

# Save the index and the chunks
faiss.write_index(index, '../vector_store/faiss_index/hr_knowledge.index')
with open('../vector_store/faiss_index/chunks.json', 'w') as f:
    json.dump(chunks, f)

print(f"FAISS index built and saved with {index.ntotal} vectors.")


## 4. The Retriever & RAG Pipeline

In [ ]:
def retrieve_context(query, top_k=2):
    query_emb = embedding_model.encode([query])
    distances, indices = index.search(query_emb, top_k)
    retrieved_chunks = [chunks[i] for i in indices[0]]
    return "\n\n".join(retrieved_chunks)

def generate_grounded_response(query):
    context = retrieve_context(query)
    
    prompt = f"""You are an expert AI Hiring Assistant. 
Use the following pieces of retrieved context to answer the candidate's question. 
If you don't know the answer based on the context, just say that you don't know. 
Keep the answer concise, professional, and actionable.

Context:
{context}

Question:
{query}

Helpful Answer:
"""
    
    if not api_key:
        return f"[GEMINI API KEY MISSING] I would have generated an answer using this context:\n\n{context}"
        
    try:
        model = genai.GenerativeModel('gemini-1.5-flash')
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Error generating response: {e}"

# Test the pipeline
test_query = "What font should I use for my resume to pass the ATS?"
print("Query:", test_query)
print("-" * 50)
print(generate_grounded_response(test_query))

test_query_2 = "Should I include my marital status on the resume?"
print("\nQuery:", test_query_2)
print("-" * 50)
print(generate_grounded_response(test_query_2))


## 5. Save as Python Module

In [ ]:
# Write the RAG components to src/rag_pipeline.py
rag_code = """import os
import json
import faiss
from sentence_transformers import SentenceTransformer
import google.generativeai as genai

class RAGPipeline:
    def __init__(self, index_path, chunks_path):
        self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.index = faiss.read_index(index_path)
        with open(chunks_path, 'r') as f:
            self.chunks = json.load(f)
            
        api_key = os.getenv("GEMINI_API_KEY")
        if api_key:
            genai.configure(api_key=api_key)
            self.llm = genai.GenerativeModel('gemini-1.5-flash')
        else:
            self.llm = None
            
    def retrieve(self, query, top_k=2):
        query_emb = self.embedding_model.encode([query])
        distances, indices = self.index.search(query_emb, top_k)
        return "\n\n".join([self.chunks[i] for i in indices[0]])
        
    def generate_response(self, query):
        context = self.retrieve(query)
        prompt = f"""You are an expert AI Hiring Assistant. Use the context to answer the candidate's question.
Context: {context}
Question: {query}
Answer:"""
        if not self.llm:
            return f"Context retrieved:\n{context}\n\n(Gemini API key missing, cannot generate final response)"
            
        try:
            response = self.llm.generate_content(prompt)
            return response.text
        except Exception as e:
            return f"Error generating response: {e}"
"""
with open('../src/rag_pipeline.py', 'w') as f:
    f.write(rag_code)
print("Saved RAG module to src/rag_pipeline.py")


# Explainable AI

**Goal:** Provide clear, structured, and interpretable feedback on why a candidate received a certain score for a job description, highlighting strengths, weaknesses, and a learning roadmap.

## Why this module exists
A black-box AI that simply outputs "Score: 72%" is frustrating for candidates and recruiters. Explainable AI (XAI) ensures transparency. We need to explain *why* the candidate matched or missed, and what they can do to improve.

## How it integrates
It consumes the parsed resume JSON (from# ), the Job Description requirements (from# ), and the semantic scores (from# & 5). It outputs a comprehensive Feedback JSON/Markdown report, which is displayed in the Streamlit UI (# ).

## Algorithms used
- **Set Operations**: For calculating Matched vs Missing Skills.
- **Rule-based Heuristics**: To evaluate strengths (e.g., years of experience exceeding requirements) and weaknesses (e.g., missing mandatory skills).
- **LLM Synthesis (Gemini)**: To generate a customized "Learning Roadmap" based on the missing skills and gaps.

## Inputs & Outputs
- **Input**: Candidate Profile JSON, Job Description Profile JSON.
- **Output**: Explainable Feedback JSON/Markdown Report.


In [1]:
import os
import json
import google.generativeai as genai

# Ensure directories exist
os.makedirs('../reports/Module_8', exist_ok=True)

api_key = os.getenv("GEMINI_API_KEY")
if api_key:
    genai.configure(api_key=api_key)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_5784\2099020784.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## 1. Load Dummy Data for XAI Demonstration

In [2]:
# Dummy Job Description
job_desc = {
    "title": "Data Scientist",
    "required_skills": ["python", "machine learning", "pandas", "scikit-learn", "deep learning", "sql", "aws"],
    "min_experience_years": 3,
    "education": "Master's Degree"
}

# Dummy Parsed Resume (simulating Module 6 output)
candidate_resume = {
    "name": "John Doe",
    "skills": "python, machine learning, pandas, sql, java",
    "experience_years": 4, # derived heuristically
    "education": "M.S. in Computer Science",
    "final_score": 75.4
}


## 2. Explainable AI Logic

In [3]:
def generate_explainable_feedback(candidate, jd):
    feedback = {
        "candidate_name": candidate["name"],
        "job_title": jd["title"],
        "overall_match_score": candidate["final_score"],
        "matched_skills": [],
        "missing_skills": [],
        "strengths": [],
        "weaknesses": [],
        "recommendations": [],
        "learning_roadmap": ""
    }
    
    # 1. Skill Match Analysis
    resume_skills = [s.strip().lower() for s in candidate["skills"].split(',')]
    jd_skills = [s.lower() for s in jd["required_skills"]]
    
    for skill in jd_skills:
        if any(skill in rs or rs in skill for rs in resume_skills):
            feedback["matched_skills"].append(skill)
        else:
            feedback["missing_skills"].append(skill)
            
    # 2. Strengths and Weaknesses
    if len(feedback["matched_skills"]) / len(jd_skills) > 0.7:
        feedback["strengths"].append("Strong alignment with core technical skills.")
    else:
        feedback["weaknesses"].append("Lacks several core technical skills required for this role.")
        
    if candidate["experience_years"] >= jd["min_experience_years"]:
        feedback["strengths"].append(f"Meets or exceeds the required experience of {jd['min_experience_years']} years.")
    else:
        feedback["weaknesses"].append(f"Falls short of the required {jd['min_experience_years']} years of experience.")
        
    # 3. Recommendations
    if feedback["missing_skills"]:
        feedback["recommendations"].append(f"Consider acquiring certifications or building projects using: {', '.join(feedback['missing_skills'][:3])}.")
    if candidate["experience_years"] < jd["min_experience_years"]:
        feedback["recommendations"].append("Focus on highlighting relevant personal projects to compensate for lack of formal experience.")

    # 4. Learning Roadmap (LLM Generated)
    roadmap_prompt = f"""Generate a short, 3-step learning roadmap for a candidate trying to become a {jd['title']}.
They are missing the following skills: {', '.join(feedback['missing_skills'])}.
Keep it concise and actionable.
"""
    if api_key:
        try:
            model = genai.GenerativeModel('gemini-1.5-flash')
            response = model.generate_content(roadmap_prompt)
            feedback["learning_roadmap"] = response.text
        except Exception as e:
            feedback["learning_roadmap"] = "Error generating roadmap."
    else:
        feedback["learning_roadmap"] = f"1. Take an online course covering {', '.join(feedback['missing_skills'][:2])}.\n2. Build a portfolio project integrating these skills.\n3. Contribute to open source."
        
    return feedback

feedback_report = generate_explainable_feedback(candidate_resume, job_desc)
print(json.dumps(feedback_report, indent=2))

# Save the report
with open('../reports/Module_8/explainable_feedback.json', 'w') as f:
    json.dump(feedback_report, f, indent=2)


{
  "candidate_name": "John Doe",
  "job_title": "Data Scientist",
  "overall_match_score": 75.4,
  "matched_skills": [
    "python",
    "machine learning",
    "pandas",
    "sql"
  ],
  "missing_skills": [
    "scikit-learn",
    "deep learning",
    "aws"
  ],
  "strengths": [
    "Meets or exceeds the required experience of 3 years."
  ],
  "weaknesses": [
    "Lacks several core technical skills required for this role."
  ],
  "recommendations": [
    "Consider acquiring certifications or building projects using: scikit-learn, deep learning, aws."
  ],
  "learning_roadmap": "1. Take an online course covering scikit-learn, deep learning.\n2. Build a portfolio project integrating these skills.\n3. Contribute to open source."
}


## 3. Save as Python Module

In [4]:
# Write the XAI components to src/explainable_ai.py
xai_code = """import os
import json
import google.generativeai as genai

class ExplainableAI:
    def __init__(self):
        api_key = os.getenv("GEMINI_API_KEY")
        if api_key:
            genai.configure(api_key=api_key)
            self.llm = genai.GenerativeModel('gemini-1.5-flash')
        else:
            self.llm = None
            
    def generate_feedback(self, candidate, jd):
        feedback = {
            "overall_match_score": candidate.get("final_score", 0),
            "matched_skills": [],
            "missing_skills": [],
            "strengths": [],
            "weaknesses": [],
            "recommendations": [],
            "learning_roadmap": ""
        }
        
        resume_skills = [s.strip().lower() for s in candidate.get("skills", "").split(',')]
        jd_skills = [s.lower() for s in jd.get("required_skills", [])]
        
        for skill in jd_skills:
            if any(skill in rs or rs in skill for rs in resume_skills):
                feedback["matched_skills"].append(skill)
            else:
                feedback["missing_skills"].append(skill)
                
        if len(feedback["matched_skills"]) / max(1, len(jd_skills)) > 0.7:
            feedback["strengths"].append("Strong alignment with core technical skills.")
        else:
            feedback["weaknesses"].append("Lacks several core technical skills required for this role.")
            
        if candidate.get("experience_years", 0) >= jd.get("min_experience_years", 0):
            feedback["strengths"].append(f"Meets the required experience of {jd.get('min_experience_years', 0)} years.")
        else:
            feedback["weaknesses"].append(f"Falls short of the required {jd.get('min_experience_years', 0)} years of experience.")
            
        if feedback["missing_skills"]:
            feedback["recommendations"].append(f"Consider acquiring certifications or building projects using: {', '.join(feedback['missing_skills'][:3])}.")
            
        roadmap_prompt = f"Generate a short, 3-step learning roadmap for a candidate trying to become a {jd.get('title', 'Professional')}. They are missing the following skills: {', '.join(feedback['missing_skills'])}. Keep it concise."
        
        if self.llm:
            try:
                response = self.llm.generate_content(roadmap_prompt)
                feedback["learning_roadmap"] = response.text
            except:
                feedback["learning_roadmap"] = "Error generating roadmap."
        else:
            feedback["learning_roadmap"] = f"1. Take a course on {', '.join(feedback['missing_skills'][:2])}.\n2. Build a project.\n3. Practice interview questions."
            
        return feedback
"""
with open('../src/explainable_ai.py', 'w') as f:
    f.write(xai_code)
print("Saved XAI module to src/explainable_ai.py")


Saved XAI module to src/explainable_ai.py


# Conversational AI

**Goal:** Create a conversational interface that allows candidates to ask natural language questions about their resume evaluation, integrating the RAG pipeline and retaining conversation history.

## Why this module exists
Instead of just handing the candidate a static report, we want to allow them to engage in a dialogue. They can ask "Why is my score low?" or "Which projects should I build?". This makes the AI Hiring Assistant feel like a true mentor.

## How it integrates
It leverages the `RAGPipeline` (from# ) to ground answers in HR best practices, and uses the candidate's `explainable feedback.json` (from# ) as personalized context. It uses LangChain's memory or a custom history buffer to maintain conversation history.

## Algorithms used
- **Generative AI Chat API**: For multi-turn conversations.
- **Context Injection**: Pre-loading the candidate's feedback into the system prompt.

## Inputs & Outputs
- **Input**: User Queries, Candidate Feedback JSON, RAG Knowledge Base.
- **Output**: Chat responses.


In [1]:
import os
import json
import google.generativeai as genai
import sys

# Append src to path to import local modules
sys.path.append('../src')
from rag_pipeline import RAGPipeline

os.makedirs('../reports/Module_9', exist_ok=True)

api_key = os.getenv("GEMINI_API_KEY")
if api_key:
    genai.configure(api_key=api_key)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_32296\1735066643.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## 1. Load RAG and Candidate Context

In [2]:
# Load the RAG Pipeline
rag = RAGPipeline('../vector_store/faiss_index/hr_knowledge.index', '../vector_store/faiss_index/chunks.json')

# Load the candidate feedback from Module 8
with open('../reports/Module_8/explainable_feedback.json', 'r') as f:
    candidate_feedback = json.load(f)
    
print(f"Loaded context for: {candidate_feedback['candidate_name']}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded context for: John Doe


## 2. Conversational Agent Logic

In [3]:
class ConversationalAgent:
    def __init__(self, candidate_feedback, rag_pipeline):
        self.feedback = candidate_feedback
        self.rag = rag_pipeline
        self.history = []
        
        # We define a strong system prompt
        self.system_prompt = f"""You are an expert AI Hiring Mentor. You are chatting with {self.feedback['candidate_name']}.
They applied for the role of {self.feedback['job_title']}.
Their overall match score is {self.feedback['overall_match_score']}%.
Matched skills: {', '.join(self.feedback['matched_skills'])}
Missing skills: {', '.join(self.feedback['missing_skills'])}
Strengths: {', '.join(self.feedback['strengths'])}
Weaknesses: {', '.join(self.feedback['weaknesses'])}

Your job is to answer their questions based on their resume feedback and standard ATS best practices.
Be encouraging but honest."""

        if api_key:
            model = genai.GenerativeModel('gemini-1.5-flash')
            self.chat = model.start_chat(history=[])
        else:
            self.chat = None
            
    def ask(self, query):
        self.history.append({"role": "user", "content": query})
        
        # 1. Retrieve HR knowledge from RAG
        hr_context = self.rag.retrieve(query)
        
        # 2. Build the augmented prompt
        augmented_prompt = f"""{self.system_prompt}

Relevant HR Context:
{hr_context}

Candidate Question: {query}
"""
        if not self.chat:
            reply = f"(No API Key) Simulated answer based on context:\nHR Info: {hr_context[:100]}...\nCandidate missing: {self.feedback['missing_skills']}"
        else:
            try:
                response = self.chat.send_message(augmented_prompt)
                reply = response.text
            except Exception as e:
                reply = f"Error generating chat: {e}"
                
        self.history.append({"role": "assistant", "content": reply})
        return reply

agent = ConversationalAgent(candidate_feedback, rag)


## 3. Test the Chatbot

In [4]:
queries = [
    "Why is my score only 75%?",
    "Which projects should I build to improve my chances?",
    "How should I format my resume to pass the ATS?"
]

chat_transcript = []

for q in queries:
    print(f"User: {q}")
    ans = agent.ask(q)
    print(f"AI: {ans}\n")
    print("-" * 50)
    chat_transcript.append({"user": q, "ai": ans})

# Save transcript
with open('../reports/Module_9/chat_transcript.json', 'w') as f:
    json.dump(chat_transcript, f, indent=2)


User: Why is my score only 75%?


AI: (No API Key) Simulated answer based on context:
HR Info: When describing work experience, always start bullet points with strong action verbs such as Develop...
Candidate missing: ['scikit-learn', 'deep learning', 'aws']

--------------------------------------------------
User: Which projects should I build to improve my chances?
AI: (No API Key) Simulated answer based on context:
HR Info: When describing work experience, always start bullet points with strong action verbs such as Develop...
Candidate missing: ['scikit-learn', 'deep learning', 'aws']

--------------------------------------------------
User: How should I format my resume to pass the ATS?
AI: (No API Key) Simulated answer based on context:
HR Info: To pass the ATS (Applicant Tracking System), candidates must avoid using complex tables, columns, or...
Candidate missing: ['scikit-learn', 'deep learning', 'aws']

--------------------------------------------------


## 4. Save as Python Module

In [5]:
# Write the Conversational AI components to src/conversational_ai.py
chat_code = """import os
import google.generativeai as genai

class ConversationalAgent:
    def __init__(self, candidate_feedback, rag_pipeline):
        self.feedback = candidate_feedback
        self.rag = rag_pipeline
        self.history = []
        
        self.system_prompt = f"You are an AI Hiring Mentor chatting with {self.feedback.get('candidate_name', 'the candidate')} applying for {self.feedback.get('job_title', 'a job')}. Their score is {self.feedback.get('overall_match_score', 0)}%. Missing skills: {', '.join(self.feedback.get('missing_skills', []))}."
        
        api_key = os.getenv("GEMINI_API_KEY")
        if api_key:
            genai.configure(api_key=api_key)
            self.model = genai.GenerativeModel('gemini-1.5-flash')
            self.chat = self.model.start_chat(history=[])
        else:
            self.chat = None
            
    def ask(self, query):
        self.history.append({"role": "user", "content": query})
        hr_context = self.rag.retrieve(query)
        augmented_prompt = f"{self.system_prompt}\nHR Context:\n{hr_context}\n\nQuestion: {query}"
        
        if not self.chat:
            reply = "I'm offline (no API key). You should learn " + ", ".join(self.feedback.get("missing_skills", []))
        else:
            try:
                response = self.chat.send_message(augmented_prompt)
                reply = response.text
            except Exception as e:
                reply = f"Error: {e}"
                
        self.history.append({"role": "assistant", "content": reply})
        return reply
"""
with open('../src/conversational_ai.py', 'w') as f:
    f.write(chat_code)
print("Saved Chat module to src/conversational_ai.py")


Saved Chat module to src/conversational_ai.py
